In [1]:
# Nếu chạy Colab L4, cell này chạy 1 lần là đủ.
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers

In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

In [3]:
# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

required_cols = ["essay_id", "essay_set", "essay", "domain1_score", "domain1_score_norm"]

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = set(required_cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

train_df.head()

,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [5]:
ESSAY_SET = 1

p1_train = train_df[train_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_val   = val_df[val_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_test  = test_df[test_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)

print("P1 train:", p1_train.shape)
print("P1 val:  ", p1_val.shape)
print("P1 test: ", p1_test.shape)

p1_train[["essay_id", "essay_set", "domain1_score", "domain1_score_norm"]].head()

P1 train: (1248, 5)
P1 val:   (180, 5)
P1 test:  (355, 5)


,essay_id,essay_set,domain1_score,domain1_score_norm
0,1311,1,10.0,0.8
1,735,1,9.0,0.7
2,1152,1,8.0,0.6
3,1314,1,10.0,0.8
4,1645,1,11.0,0.9


In [6]:
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

Y_MIN, Y_MAX = ASAP_SCORE_RANGES[ESSAY_SET]
Y_MIN, Y_MAX

(2, 12)

In [7]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
    }

In [8]:
def sample_pairs(df, num_pairs=1000, seed=42):
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).tolist()
    n = len(essay_ids)

    pairs = set()

    # First pass: make sure each essay appears at least once if possible
    shuffled = essay_ids.copy()
    rng.shuffle(shuffled)

    for i in range(0, len(shuffled) - 1, 2):
        a, b = shuffled[i], shuffled[i + 1]
        if a > b:
            a, b = b, a
        pairs.add((a, b))

        if len(pairs) >= num_pairs:
            break

    # Fill remaining pairs randomly
    while len(pairs) < num_pairs:
        a, b = rng.choice(essay_ids, size=2, replace=False)
        a, b = int(a), int(b)

        if a > b:
            a, b = b, a

        pairs.add((a, b))

    return pd.DataFrame(list(pairs), columns=["essay_id_1", "essay_id_2"])


M_TRAIN = 5000   # debug first
M_VAL   = 100    # optional validation check

train_pairs = sample_pairs(
    p1_train,
    num_pairs=M_TRAIN,
    seed=SEED
)
val_pairs   = sample_pairs(p1_val, num_pairs=M_VAL, seed=SEED)

train_pairs.head(), train_pairs.shape

(   essay_id_1  essay_id_2
 0         168        1682
 1           3         797
 2         624         915
 3         245         330
 4         659        1367,
 (5000, 2))

In [9]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.2


In [10]:
P1_RUBRIC_COMPACT = """
Evaluate the overall quality of an 8th-grade persuasive letter about the effects of computers on people.

Prefer the essay that better:
- states a clear position,
- gives persuasive reasons and specific details,
- is well organized and coherent,
- uses transitions effectively,
- shows awareness of a local newspaper audience,
- is fluent and readable,
- controls grammar, usage, and mechanics.

Choose "tie" only if the two essays are genuinely very close in overall quality.
""".strip()

In [11]:
P1_PROMPT_TEXT = """
More and more people use computers, but not everyone agrees that this benefits society.
Those who support advances in technology believe that computers have a positive effect on people.
They teach hand-eye coordination, give people the ability to learn about faraway places and people,
and even allow people to talk online with other people. Others have different ideas.
Some experts are concerned that people are spending too much time on their computers and less time
exercising, enjoying nature, and interacting with family and friends.

Write a letter to your local newspaper in which you state your opinion on the effects computers have
on people. Persuade the readers to agree with you.
""".strip()

In [12]:
def build_pairwise_prompt(essay1, essay2):
    return f"""
Compare two 8th-grade persuasive essays for overall writing quality.

# Writing Prompt
{P1_PROMPT_TEXT}

# Rubric
{P1_RUBRIC_COMPACT}

# Essay 1
{essay1}

# Essay 2
{essay2}

Return JSON only:
{{
  "reasoning": "one concise sentence",
  "preference": "essay1" or "essay2" or "tie"
}}
""".strip()

In [13]:
def extract_json(text):
    """
    Robustly extract:
    {
      "reasoning": "...",
      "preference": "essay1" | "essay2" | "tie"
    }
    """
    text = text.strip()

    # 1. Direct JSON parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "preference" in obj:
            return obj
    except Exception:
        pass

    # 2. Extract first JSON-like block
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict) and "preference" in obj:
                return obj
        except Exception:
            pass

    # 3. Regex preference field from imperfect JSON/text
    pref_match = re.search(
        r'["\']?preference["\']?\s*[:=]\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie)',
        text,
        flags=re.IGNORECASE
    )

    if pref_match:
        pref = pref_match.group(1).lower().replace(" ", "")
    else:
        # 4. Last-resort: only inspect the end of output
        tail = text[-300:].lower()

        if re.search(r'\bpreference\b.*\bessay\s*1\b|\bpreference\b.*\bessay1\b', tail):
            pref = "essay1"
        elif re.search(r'\bpreference\b.*\bessay\s*2\b|\bpreference\b.*\bessay2\b', tail):
            pref = "essay2"
        elif "tie" in tail or "equal" in tail or "same quality" in tail:
            pref = "tie"
        else:
            pref = "tie"

    return {
        "reasoning": text[:300],
        "preference": pref
    }


tokenizer.padding_side = "left"

@torch.no_grad()
def call_local_llm(prompts, max_new_tokens=80, temperature=0.0):
    """
    Batched local LLM inference.

    Memory-saving changes:
    - Uses max_length=2048 instead of 4096.
    - Keeps batch inference.
    - Deletes large tensors before returning.
    """
    input_texts = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            input_text = prompt

        input_texts.append(input_text)

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to(llm.device)

    output_ids = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]

    texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    results = [extract_json(t) for t in texts]

    del inputs
    del output_ids
    del generated_ids

    return results

In [14]:
# Quick smoke test
sample_essay_1 = p1_train.iloc[0]["essay"]
sample_essay_2 = p1_train.iloc[1]["essay"]

test_prompt = build_pairwise_prompt(sample_essay_1, sample_essay_2)
call_local_llm([test_prompt])

[{'reasoning': 'Essay 1 presents more persuasive reasons with specific details and examples, and is more coherently organized.',
  'preference': 'essay1'}]

In [15]:
def preference_to_label(preference):
    pref = str(preference).lower().strip()
    pref = pref.replace(" ", "")
    pref = pref.replace("_", "")
    pref = pref.replace('"', "")
    pref = pref.replace("'", "")

    if pref in ["essay1", "1"]:
        return 1.0
    if pref in ["essay2", "2"]:
        return 0.0
    if pref in ["tie", "equal", "same", "both"]:
        return 0.5

    return 0.5

def debias_label(label_forward, label_reverse):
    """
    Forward: essay_id_1 as Essay 1, essay_id_2 as Essay 2.
    Reverse: essay_id_2 as Essay 1, essay_id_1 as Essay 2.

    If consistent:
      label_forward == 1 - label_reverse

    Else:
      tie.
    """
    if np.isclose(label_forward, 1.0 - label_reverse):
        return label_forward
    return 0.5

In [16]:
def generate_pairwise_judgments(
    df,
    pairs_df,
    debias=True,
    batch_size=8,
    max_new_tokens=32
):
    """
    Generate pairwise LLM judgments.

    Memory-saving changes:
    - Keeps batch_size=8.
    - Clears temporary prompt/result variables each batch.
    - Calls torch.cuda.empty_cache() every 25 batches.
    """
    essay_map = df.set_index("essay_id")["essay"].to_dict()
    rows = []

    pair_rows = list(pairs_df.itertuples(index=False))

    for start in tqdm(range(0, len(pair_rows), batch_size)):
        batch = pair_rows[start:start + batch_size]

        forward_prompts = []
        reverse_prompts = []
        meta = []

        for row in batch:
            id1 = int(row.essay_id_1)
            id2 = int(row.essay_id_2)

            essay1 = essay_map[id1]
            essay2 = essay_map[id2]

            forward_prompts.append(build_pairwise_prompt(essay1, essay2))

            if debias:
                reverse_prompts.append(build_pairwise_prompt(essay2, essay1))

            meta.append((id1, id2))

        forward_results = call_local_llm(
            forward_prompts,
            max_new_tokens=max_new_tokens,
            temperature=0.0
        )

        if debias:
            reverse_results = call_local_llm(
                reverse_prompts,
                max_new_tokens=max_new_tokens,
                temperature=0.0
            )
        else:
            reverse_results = [None] * len(forward_results)

        for (id1, id2), result_forward, result_reverse in zip(meta, forward_results, reverse_results):
            label_forward = preference_to_label(result_forward.get("preference", "tie"))

            if debias:
                label_reverse = preference_to_label(result_reverse.get("preference", "tie"))
                final_label = debias_label(label_forward, label_reverse)

                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": final_label,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": result_reverse.get("preference", "tie"),
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": result_reverse.get("reasoning", ""),
                })
            else:
                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": label_forward,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": None,
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": None,
                })

        del batch
        del forward_prompts
        del reverse_prompts
        del forward_results
        del reverse_results
        del meta

        batch_idx = start // batch_size
        if torch.cuda.is_available() and batch_idx % 25 == 0:
            torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

tie = true

In [17]:
train_judgments = generate_pairwise_judgments(
    df=p1_train,
    pairs_df=train_pairs,
    debias=True,
    batch_size=8,
    max_new_tokens=32
)

train_judgments.head()

  0%|          | 0/625 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
train_judgments["label"].value_counts(dropna=False)

used_ids = set(train_judgments["essay_id_1"]) | set(train_judgments["essay_id_2"])

print("Total P1 train essays:", len(p1_train))
print("Essays appearing in comparisons:", len(used_ids))
print("Coverage:", len(used_ids) / len(p1_train))

pair_count = pd.concat([
    train_judgments["essay_id_1"],
    train_judgments["essay_id_2"]
]).value_counts()

print("\nPair count per essay:")
print(pair_count.describe())

def check_consistency(row):
    lf = preference_to_label(row["pref_forward"])
    lr = preference_to_label(row["pref_reverse"])
    return np.isclose(lf, 1.0 - lr)

consistency_rate = train_judgments.apply(check_consistency, axis=1).mean()

print("Consistency rate:", consistency_rate)
print("Inconsistency rate:", 1 - consistency_rate)

display(
    train_judgments.assign(
        consistent=train_judgments.apply(check_consistency, axis=1)
    )[["essay_id_1", "essay_id_2", "pref_forward", "pref_reverse", "label", "consistent"]].head(20)
)

In [ ]:
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

def encode_essays(df, text_col="essay", batch_size=32):
    embeddings = embedder.encode(
        df[text_col].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings.astype(np.float32)


train_embeddings = encode_essays(p1_train)
val_embeddings   = encode_essays(p1_val)
test_embeddings  = encode_essays(p1_test)

train_embeddings.shape, val_embeddings.shape, test_embeddings.shape

In [ ]:
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

In [ ]:
def train_ranknet(
    df,
    embeddings,
    judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
    device=DEVICE
):
    essay_ids = df["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].isin(id_to_idx) &
        judgments["essay_id_2"].isin(id_to_idx)
    ].copy()

    assert len(pair_df) > 0, "No valid pairwise judgments for this dataframe."

    idx_i = pair_df["essay_id_1"].map(id_to_idx).values
    idx_j = pair_df["essay_id_2"].map(id_to_idx).values
    labels = pair_df["label"].values.astype(np.float32)

    x = torch.tensor(embeddings, dtype=torch.float32, device=device)

    model = RankNet(
        input_dim=embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    loss_fn = nn.BCELoss()

    n = len(pair_df)

    for epoch in range(1, epochs + 1):
        order = np.random.permutation(n)
        epoch_losses = []

        model.train()

        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]

            bi = torch.tensor(idx_i[batch], dtype=torch.long, device=device)
            bj = torch.tensor(idx_j[batch], dtype=torch.long, device=device)
            y = torch.tensor(labels[batch], dtype=torch.float32, device=device)

            s_i = model(x[bi])
            s_j = model(x[bj])

            pred = torch.sigmoid(s_i - s_j)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | loss = {np.mean(epoch_losses):.4f}")

    return model

In [ ]:
ranknet = train_ranknet(
    df=p1_train,
    embeddings=train_embeddings,
    judgments=train_judgments,
    epochs=150,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=512,
    dropout=0.1,
)

In [ ]:
@torch.no_grad()
def predict_latent_scores(model, embeddings, device=DEVICE):
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


train_latent = predict_latent_scores(ranknet, train_embeddings)
val_latent   = predict_latent_scores(ranknet, val_embeddings)
test_latent  = predict_latent_scores(ranknet, test_embeddings)

train_latent[:5], val_latent[:5], test_latent[:5]

In [ ]:
def fit_latent_mapper(train_latent, y_min, y_max):
    s_min = float(np.min(train_latent))
    s_max = float(np.max(train_latent))

    def mapper(latent_scores):
        latent_scores = np.asarray(latent_scores, dtype=float)

        if np.isclose(s_min, s_max):
            mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2)
        else:
            mapped = (latent_scores - s_min) / (s_max - s_min)
            mapped = mapped * (y_max - y_min) + y_min

        mapped = np.rint(mapped)
        mapped = np.clip(mapped, y_min, y_max)

        return mapped.astype(int)

    return mapper


score_mapper = fit_latent_mapper(train_latent, Y_MIN, Y_MAX)

p1_train_pred = score_mapper(train_latent)
p1_val_pred   = score_mapper(val_latent)
p1_test_pred  = score_mapper(test_latent)

p1_test_pred[:20]

In [ ]:
train_metrics = compute_metrics(
    y_true=p1_train["domain1_score"],
    y_pred=p1_train_pred
)

val_metrics = compute_metrics(
    y_true=p1_val["domain1_score"],
    y_pred=p1_val_pred
)

test_metrics = compute_metrics(
    y_true=p1_test["domain1_score"],
    y_pred=p1_test_pred
)

pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "val", **val_metrics},
    {"split": "test", **test_metrics},
])

In [ ]:
p1_test_results = p1_test[[
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm"
]].copy()

p1_test_results["latent_score"] = test_latent
p1_test_results["pred_score"] = p1_test_pred
p1_test_results["abs_error"] = np.abs(
    p1_test_results["domain1_score"] - p1_test_results["pred_score"]
)

p1_test_results[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score"
]].head(20)

In [ ]:
# Worst predictions
p1_test_results.sort_values("abs_error", ascending=False)[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
    "essay"
]].head(10)